# ViT-S/16 Room Classification — Colab runner

**How to use (VS Code + Google Colab extension):** open this file, click **Select Kernel** (top-right) → **Colab** → pick a **GPU** runtime → sign in with Google. Then run the cells top to bottom.

It also works in the browser (colab.research.google.com): set Runtime → Change runtime type → **T4 GPU**.

> The kernel runs on a **remote Colab VM**, not your laptop — so the cells below clone the code from GitHub and read the dataset from your Google Drive.

## 1. Check the GPU

In [2]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## 2. Get the code (clone from GitHub) and install deps
`torch`/`torchvision` are preinstalled on Colab, so only the light extras get installed.

In [2]:
import os

REPO_DIR = '/content/room-classification'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/SilviuBR24/room-classification.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}/vit_s16_baseline
!pip install -q -r requirements.txt
print('Code ready.')

Cloning into '/content/room-classification'...
remote: Enumerating objects: 30661, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 30661 (delta 9), reused 41 (delta 5), pack-reused 30612 (from 1)
Receiving objects: 100% (30661/30661), 336.64 MiB | 25.48 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Updating files: 100% (30635/30635), done.
/content/room-classification/vit_s16_baseline
Code ready.


## 3. Mount Google Drive (dataset + persistent checkpoints)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Point to your dataset on Drive and verify it
⚠️ **Set `DRIVE_DATASET` to the exact folder where you uploaded the dataset.** It must contain `train/` and `eval/`, each with the 6 class folders. The assert below fails loudly with the missing path if it's wrong.

In [6]:
import os

# Your dataset lives at: My Drive / room-classification / dataset
DRIVE_DATASET = '/content/drive/MyDrive/Dissertation_Thesis/room-classification/dataset'

EXPECTED = ['bathroom', 'bedroom', 'dining_room', 'entrance_hall', 'kitchen', 'living_room']
for split in ['train', 'eval']:
    p = os.path.join(DRIVE_DATASET, split)
    assert os.path.isdir(p), f'Not found: {p}  ->  fix DRIVE_DATASET above'
    classes = sorted(d for d in os.listdir(p) if os.path.isdir(os.path.join(p, d)))
    counts = {c: len(os.listdir(os.path.join(p, c))) for c in classes}
    assert classes == EXPECTED, f'{split}: expected {EXPECTED}, got {classes}'
    print(f'{split:5s}:', counts)
print('Dataset OK.')

## 5. Unzip the split dataset (3500 labeled / 1500 unlabeled) + build the Colab config
Copies the single `dataset_split.zip` from Drive to local SSD, unzips it (backslash-safe, since the zip was made on Windows), and writes `config_colab.yaml` pointing at the **3500-labeled** split. `run_name` is set to `vit_s16_3500` so this run's folder is distinct from the earlier 5000-image baseline. Checkpoints still go to Drive.

> The `unlabeled/` 1500-per-class set is extracted too — it is **not** used for supervised training here; it is reserved for the later self-labeling stage.

In [ ]:
import os, time, shutil, zipfile, yaml

# Split dataset (3500 labeled / 1500 unlabeled / 100 eval), uploaded as ONE zip on Drive
DRIVE_ZIP = '/content/drive/MyDrive/Dissertation_Thesis/dataset_split.zip'
LOCAL_ZIP = '/content/dataset_split.zip'
DATA_ROOT = '/content/dataset_split'

if not os.path.isdir(os.path.join(DATA_ROOT, 'train')):
    if not os.path.exists(LOCAL_ZIP):
        t0 = time.time(); shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
        print(f'Copied zip Drive->local in {time.time()-t0:.0f}s')
    t1 = time.time()
    with zipfile.ZipFile(LOCAL_ZIP) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/')   # Windows backslash -> /
            if name.endswith('/'):
                continue
            target = os.path.join('/content', name)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)
    print(f'Extracted (backslash-safe) in {time.time()-t1:.0f}s')
else:
    print(f'{DATA_ROOT} already present, skipping.')

# verify: train=3500, unlabeled=1500, eval=100 per class
for sub in ['train', 'unlabeled', 'eval']:
    p = os.path.join(DATA_ROOT, sub)
    counts = {c: len(os.listdir(os.path.join(p, c))) for c in sorted(os.listdir(p))}
    print(f'{sub:9s}:', counts)

# config for the 3500-labeled run (distinct run_name)
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['run_name'] = 'vit_s16_3500'
cfg['data']['train_dir'] = DATA_ROOT + '/train'
cfg['data']['eval_dir']  = DATA_ROOT + '/eval'
cfg['paths']['output_root'] = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
cfg['training']['batch_size'] = 64
cfg['training']['num_workers'] = os.cpu_count()
with open('config_colab.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('\nconfig_colab.yaml | run_name =', cfg['run_name'], '| train =', cfg['data']['train_dir'])

## 6. Train
Runs the full 50-epoch supervised baseline. Checkpoints (`best_model.pt`, `last_checkpoint.pt`) land in the run folder on Drive. If the session disconnects, re-run cells 2–5, then resume with the `--resume` line printed at the end.

In [6]:
!python train.py --config config_colab.yaml

[train] New run folder: /content/drive/MyDrive/Dissertation_Thesis/dissertation_runs/2026-06-27_10-54_vit_s16_baseline
2026-06-27 10:54:15 | INFO    | Device: cuda
2026-06-27 10:54:15 | INFO    | Run directory: /content/drive/MyDrive/Dissertation_Thesis/dissertation_runs/2026-06-27_10-54_vit_s16_baseline
2026-06-27 10:54:16 | INFO    | Train images: 30000 | Eval images: 600 | classes: {'bathroom': 0, 'bedroom': 1, 'dining_room': 2, 'entrance_hall': 3, 'kitchen': 4, 'living_room': 5}
2026-06-27 10:54:16 | INFO    | Model: ViT-S/16 from scratch | trainable params: 21,691,014
2026-06-27 10:54:16 | INFO    | Starting training: epochs 1..50, device=cuda, amp=True
2026-06-27 10:57:29 | INFO    | Epoch   1/50 | lr=6.00e-05 | train_loss=1.7101 train_acc=0.2806 | eval_loss=1.6569 eval_acc=0.3350 | best_acc=0.3350 | 192.4s  <-- new best
2026-06-27 11:00:52 | INFO    | Epoch   2/50 | lr=1.20e-04 | train_loss=1.6379 train_acc=0.3337 | eval_loss=1.5739 eval_acc=0.3967 | best_acc=0.3967 | 203.3s  <-

## 7. Evaluate the best checkpoint
Finds the most recent run on Drive and evaluates its `best_model.pt` (confusion matrix, per-class report, predictions). Add `--save-embeddings` for t-SNE later.

In [ ]:
import glob, os

REPO = '/content/room-classification'
PROJECT = REPO + '/vit_s16_baseline'
RUNS_DIR = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
EVAL_DIR = REPO + '/dataset/eval'        # 100/class, from the git clone (same eval set)

if not os.path.isdir(PROJECT):
    !git clone https://github.com/SilviuBR24/room-classification.git {REPO}
os.chdir(PROJECT)                        # evaluate.py imports from src/, must run from here

# evaluates the MOST RECENT run; to pick a specific one, set `best` by hand
runs = sorted(glob.glob(RUNS_DIR + '/*/'))
assert runs, f'No runs found in {RUNS_DIR}'
best = os.path.join(runs[-1], 'checkpoints', 'best_model.pt')
print('Evaluating:', best, '\ncwd:', os.getcwd())
!python evaluate.py --checkpoint "{best}" --data-dir "{EVAL_DIR}"